In [ ]:
import json
import math
import re
from pathlib import Path

import pandas as pd

MODELS = Path("/projects/ml/na_mpnn/model_inputs")
CSVS = Path("../evaluation/ablation_csvs")

# Strips trailing inference/checkpoint suffixes from model names so they
# resolve back to the underlying training-config JSON file:
#   _0_1, _0_3, _0_5     - sampling temperature used at inference
#   _later               - later training checkpoint of the same run
#   _earlier_checkpoint  - earlier training checkpoint of the same run
SUFFIX = re.compile(r"(_later|_earlier_checkpoint|_0_\d)+$")

# Hyperparameters that always differ between models but are not meaningful
# ablation knobs - hide them from the diff table.
SKIP_KEYS = {"BASE_FOLDER", "PREV_CHECKPOINT"}

def load_config(model_name):
    """
    Load the training-config JSON for a model, ignoring inference suffixes.
    """
    base_name = SUFFIX.sub("", model_name)
    return json.loads((MODELS / f"{base_name}.json").read_text())

def diff_table(model_names):
    """
    Build a DataFrame showing only the hyperparameters that differ between
    the given models. Rows are model names in the order supplied;
    columns are the varying keys.
    """
    configs = {name: load_config(name) for name in dict.fromkeys(model_names)}

    # Take the union of all keys across configs, then keep only those whose
    # values are not identical across every config.
    all_keys = {key for cfg in configs.values() for key in cfg} - SKIP_KEYS
    varying_keys = sorted(
        key for key in all_keys
        if len({json.dumps(cfg.get(key)) for cfg in configs.values()}) > 1
    )

    rows = {name: {key: cfg.get(key) for key in varying_keys} for name, cfg in configs.items()}
    return pd.DataFrame(rows).T

def round_half_up(value, places=2):
    """
    Round to `places` decimal places using schoolbook rounding: 0.5 always
    rounds up. Unlike pandas/numpy `.round()`, which uses banker's rounding
    (round-half-to-even). Assumes non-negative values.
    """
    if pd.isna(value):
        return value
    multiplier = 10 ** places
    return math.floor(value * multiplier + 0.5) / multiplier

## Design ablation

In [ ]:
design = pd.read_csv(CSVS / "design_valid_ablation_results.csv")
design = design[design["Model"].isin(["model_v_172", "model_v_176"])]
diff_table(design["Model"].unique())

In [ ]:
groups = ["DNA", "Protein-DNA", "RNA", "Protein-RNA"]
(
    design.groupby(["Model", "Group"])["Sequence Recovery"].median()
    .unstack().reindex(columns=groups)
    .map(lambda x: round_half_up(x, places=3))
)

## Specificity ablation

In [ ]:
specificity = pd.read_csv(CSVS / "specificity_valid_ablation_results.csv")

models = [
    "model_v_186_0_5",
    "model_v_177_later_0_5",
    "model_v_184_0_5",
    "model_v_185_0_5",
]
specificity = specificity[specificity["Model"].isin(models)]
specificity["Model"] = pd.Categorical(specificity["Model"], categories=models, ordered=True)
diff_table(models)

In [ ]:
groups = ["PPM from distillation", "PPM from crystal"]
metrics = ["Mean Absolute Error", "Cross Entropy"]
(
    specificity.groupby(["Model", "Group"], observed=True)[metrics].median()
    .unstack("Group")
    .reindex(columns=pd.MultiIndex.from_product([metrics, groups]))
    .map(round_half_up)
)